In [ ]:
API_KEY = ""

# Choose a gemini model
import google.generativeai as genai
from google.colab import userdata
GOOGLE_API_KEY = API_KEY
genai.configure(api_key=GOOGLE_API_KEY)
gemini_model = genai.GenerativeModel('models/gemini-2.5-flash-lite')

# Load the structurized manual
from google.colab import drive
drive.mount('/content/drive')

file_path = "/content/drive/MyDrive/Haystack/manual.json"
import json
with open(file_path, "r", encoding="utf-8") as file:
  manual=json.load(file)

# save the contexts, titles and pages for later use
contexts = [document["context"] for document in manual]
titles = [document["title"] for document in manual]
pages = [document["page"] for document in manual]

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Mounted at /content/drive


In [ ]:
!pip install haystack-ai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 643.7/643.7 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.6/167.6 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.2/67.2 kB 4.6 MB/s eta 0:00:00


In [ ]:
# Prepare the document_store, doc_embedder and text_embedder using haystack
from haystack.document_stores.in_memory import InMemoryDocumentStore
from haystack import Document
from haystack.components.embedders import SentenceTransformersDocumentEmbedder, SentenceTransformersTextEmbedder
from haystack.components.retrievers.in_memory import InMemoryEmbeddingRetriever
from haystack.components.builders import PromptBuilder
from haystack.components.generators import HuggingFaceLocalGenerator
from haystack.utils import ComponentDevice

document_store = InMemoryDocumentStore()
docs = [Document(content = contexts[i], meta = {'title': titles[i], 'context_id': i, 'page': pages[i]}) for i in range(len(manual))]

doc_embedder = SentenceTransformersDocumentEmbedder(
    model="all-MiniLM-L6-v2",
    progress_bar=False
)
doc_embedder.warm_up()

docs_with_embeddings = doc_embedder.run(docs)
document_store.write_documents(docs_with_embeddings["documents"])

text_embedder = SentenceTransformersTextEmbedder(
    model="all-MiniLM-L6-v2",
    progress_bar=False
)
text_embedder.warm_up()

retriever = InMemoryEmbeddingRetriever(document_store)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
# load the QAs
manual_QAs_json_path = "/content/drive/MyDrive/Haystack/manual_QAs.json"
with open(manual_QAs_json_path, "r", encoding="utf-8") as qas_file:
  manual_QAs = json.load(qas_file)

In [ ]:
# define a function for rewritting a query
def rewrite_query(query, history_conversations):
  template_for_query_rewrite = f"""
You are an intelligent assistant for rewriting a question.
You need to rewrite a new question based on the history conversations.
-----------------------------------------------------------------------------------
History conversations:
{history_conversations}
-----------------------------------------------------------------------------------
New question: {query}
Rewrite the new question according to the history conversations. Make sure the rewritten new question is concise and understandable without the history conversations.
Rewrite the new question in the format below:
Rewritten Question: ...
"""
  model_rewritten_query = gemini_model.generate_content(template_for_query_rewrite)
  rewritten_query = model_rewritten_query.text.strip("Rewritten Question:")
  rewritten_query = rewritten_query.strip()
  return rewritten_query

In [ ]:
all_rewritten_queries = []
for index in range(40):
  data = manual_QAs["LDS"][index]
  queries = data['Queries']
  answers = data['Answers']
  ids = data['Context_ids']

  history_conversations = ""
  rewritten_queries = [queries[0]]
  for i in range(1, len(queries)):
    history_conversations += f"Question {i}: {queries[i-1]}\n"
    history_conversations += f"Answer {i}: {answers[i-1]}\n"
    rewritten_queries.append(rewrite_query(queries[i], history_conversations))

  all_rewritten_queries.append(rewritten_queries)

import json
with open("/content/drive/MyDrive/Haystack/LDS_rewritten_queries.json", 'w', encoding='utf-8') as f:
  json.dump(all_rewritten_queries, f, indent = 4)

In [ ]:
def similarity(text1, text2):
  text1_embedding = text_embedder.run(text1)
  text2_embedding = text_embedder.run(text2)
  v1 = torch.tensor(text1_embedding["embedding"])
  v2 = torch.tensor(text2_embedding["embedding"])
  sim = F.cosine_similarity(v1, v2, dim=0)
  return sim.item()

import torch
import torch.nn.functional as F

In [ ]:
# collect retrieval context ids using current queries
all_current_ids = []
for index in range(40):
  data = manual_QAs["LDS"][index]
  queries = data['Queries']
  ids = data['Context_ids']

  current_ids = []
  for query in queries:
    retrieval_input_embedding = text_embedder.run(query)
    docs = retriever.run(query_embedding = retrieval_input_embedding['embedding'], top_k = 20)
    current_ids.append([docs["documents"][i].meta['context_id'] for i in range(20)])

  all_current_ids.append(current_ids)

import json
with open("/content/drive/MyDrive/Haystack/LDS_current_ids.json", 'w', encoding='utf-8') as f:
  json.dump(all_current_ids, f, indent = 4)

In [ ]:
import json
with open("/content/drive/MyDrive/Haystack/LDS_rewritten_queries.json", 'r', encoding='utf-8') as f:
  all_rewritten_queries = json.load(f)

In [ ]:
# collect retrieval context ids using rewritten queries
all_rewritten_ids = []
for index in range(40):
  data = manual_QAs["LDS"][index]
  queries = all_rewritten_queries[index]
  ids = data['Context_ids']

  rewritten_ids = []
  for query in queries:
    retrieval_input_embedding = text_embedder.run(query)
    docs = retriever.run(query_embedding = retrieval_input_embedding['embedding'], top_k = 20)
    rewritten_ids.append([docs["documents"][i].meta['context_id'] for i in range(20)])

  all_rewritten_ids.append(rewritten_ids)

import json
with open("/content/drive/MyDrive/Haystack/LDS_rewritten_ids.json", 'w', encoding='utf-8') as f:
  json.dump(all_rewritten_ids, f, indent = 4)

In [ ]:
all_correct_ids = [manual_QAs["LDS"][i]['Context_ids'] for i in range(40)]

In [ ]:
all_current_scores = []
for i in range(5):
  print(i, end = ": ")
  correct_ids = all_correct_ids[i]
  current_ids = all_current_ids[i]
  current_scores = []
  for n in range(len(correct_ids)):
    print(n, end = " ")
    current_score = []
    for k in range(20):
      correct_id = correct_ids[n]
      current_id = current_ids[n][k]
      correct_context = contexts[correct_id]
      current_context = contexts[current_id]
      current_score.append(similarity(current_context, correct_context))
    current_scores.append(current_score)
  print()
  all_current_scores.append(current_scores)
with open("/content/drive/MyDrive/Haystack/LDS_current_scores.json", 'w', encoding='utf-8') as f:
  json.dump(all_current_scores, f)

0: 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 
1: 0 1 2 3 4 5 6 7 8 
2: 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 
3: 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 
4: 0 1 2 3 4 5 6 7 8 9 10 11 


In [ ]:
all_rewritten_scores = []
for i in range(40):
  print(i, end = ": ")
  correct_ids = all_correct_ids[i]
  rewritten_ids = all_rewritten_ids[i]
  rewritten_scores = []
  for n in range(len(correct_ids)):
    print(n, end = " ")
    rewritten_score = []
    for k in range(20):
      correct_id = correct_ids[n]
      rewritten_id = rewritten_ids[n][k]
      correct_context = contexts[correct_id]
      rewritten_context = contexts[rewritten_id]
      rewritten_score.append(similarity(rewritten_context, correct_context))
    rewritten_scores.append(rewritten_score)
  print()
  all_rewritten_scores.append(rewritten_scores)
with open("/content/drive/MyDrive/Haystack/LDS_rewritten_scores.json", 'w', encoding='utf-8') as f:
  json.dump(all_rewritten_scores, f)

0: 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 
1: 0 1 2 3 4 5 6 7 8 
2: 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 
3: 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 
4: 0 1 2 3 4 5 6 7 8 9 10 11 
5: 0 1 2 3 4 5 6 7 
6: 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 
7: 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 
8: 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 
9: 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 
10: 0 1 2 3 4 5 6 7 8 9 10 11 12 
11: 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 
12: 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 
13: 0 1 2 3 4 5 6 7 8 
14: 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 
15: 0 1 2 3 4 5 6 7 8 9 10 11 12 
16: 0 1 2 3 4 5 6 7 8 9 10 
17: 0 1 2 3 4 5 6 7 8 9 10 
18: 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 
19: 0 1 2 3 4 5 6 7 8 
20: 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 
21: 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 
22: 0 1 2 3 4 5 6 7 8 9 10 11 12 13 
23: 0 1 2 3 4 5 6 7

In [ ]:
for i in range(6,40):
  print(i, end = ": ")
  correct_ids = all_correct_ids[i]
  current_ids = all_current_ids[i]
  current_scores = []
  for n in range(len(correct_ids)):
    print(n, end = " ")
    current_score = []
    for k in range(20):
      correct_id = correct_ids[n]
      current_id = current_ids[n][k]
      correct_context = contexts[correct_id]
      current_context = contexts[current_id]
      current_score.append(similarity(current_context, correct_context))
    current_scores.append(current_score)
  print()
  all_current_scores.append(current_scores)
with open("/content/drive/MyDrive/Haystack/LDS_current_scores.json", 'w', encoding='utf-8') as f:
  json.dump(all_current_scores, f)

6: 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 
7: 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 
8: 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 
9: 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 
10: 0 1 2 3 4 5 6 7 8 9 10 11 12 
11: 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 
12: 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 
13: 0 1 2 3 4 5 6 7 8 
14: 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 
15: 0 1 2 3 4 5 6 7 8 9 10 11 12 
16: 0 1 2 3 4 5 6 7 8 9 10 
17: 0 1 2 3 4 5 6 7 8 9 10 
18: 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 
19: 0 1 2 3 4 5 6 7 8 
20: 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 
21: 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 
22: 0 1 2 3 4 5 6 7 8 9 10 11 12 13 
23: 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 
24: 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 
25: 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 
26: 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 
27: 0 1 2 3 

In [ ]:
with open("/content/drive/MyDrive/Haystack/LDS_current_scores.json", "r", encoding="utf-8") as file:
  all_current_scores=json.load(file)
with open("/content/drive/MyDrive/Haystack/LDS_current_ids.json", "r", encoding="utf-8") as file:
  all_current_ids=json.load(file)
all_correct_ids = [manual_QAs["LDS"][i]['Context_ids'] for i in range(40)]

In [ ]:
i = 5
correct_ids = all_correct_ids[i]
current_ids = all_current_ids[i]
current_scores = []
for n in range(len(correct_ids)):
  print(n, end = " ")
  current_score = []
  for k in range(20):
    correct_id = correct_ids[n]
    current_id = current_ids[n][k]
    correct_context = contexts[correct_id]
    current_context = contexts[current_id]
    current_score.append(similarity(current_context, correct_context))
  current_scores.append(current_score)
  print()

0 
1 
2 
3 
4 
5 
6 
7 


In [ ]:
all_current_scores = all_current_scores[0:6] + [current_scores] + all_current_scores[6:]

In [ ]:
len(all_current_scores)

40

In [ ]:
with open("/content/drive/MyDrive/Haystack/LDS_current_scores.json", 'w', encoding='utf-8') as f:
  json.dump(all_current_scores, f)